<a href="https://colab.research.google.com/github/danielandresrestrebadaleal-dev/ARC/blob/main/GEO_orbital_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Intro & Goal of this program

This program simulates the orbital profile of a payload carried by a two-stage rocket from a LEO orbit to insertion in a GEO orbit. To achieve this, the spacecraft performs a classic Hohmann transfer maneuver. The code implements realistic perturbing forces, such as the gravitational influence of the Moon, the Sun, J-type forces, etc. The goal of this program is to determine what is the associated delta-V cost and fuel mass needed to insert a payload in a GEO orbit, given specific launch parameters such as launch latitude, initial LEO altitude, etc.

In [14]:
#Install necessary packages for Google Colab. Note: hapsira is the modern version of poliastro to handle Python 3.12
!pip install hapsira "astropy<7.0"

# Code Imports
This code utilizes several Python libraries for different functionalities:

1. poliastro: defines celestial bodies and physical constants, calculates the evolution of the orbit using Cowell Propagator, and introduces the effects of solar radiation and other forces

2. astropy: handling time objects, astronomical units, etc.

In [15]:
from astropy import units as u
from hapsira.twobody import Orbit
from hapsira.core.perturbations import (
    J2_perturbation, J3_perturbation, third_body,
    radiation_pressure, atmospheric_drag_exponential
)
from hapsira.ephem import build_ephem_interpolant, Ephem
from hapsira.bodies import Earth, Moon, Sun
from hapsira.maneuver import Maneuver
from hapsira.constants import rho0_earth, H0_earth

import numpy as np
from astropy.time import Time
from astropy.coordinates import CartesianRepresentation, CartesianDifferential, SkyCoord, ICRS, GCRS
from astropy.coordinates import solar_system_ephemeris
from hapsira.twobody.propagation import CowellPropagator
from hapsira.core.propagation import func_twobody

# Interpolating values

Because this program is meant to run multiple times with different launch parameters, we provide a list of apporixmate delta-V losses during takeoff to reach a specific LEO altitude. Interpolation is then used if any delta-V values is needed that is in between any of the provided LEO altitudes. This saves us the trouble of having to make complex calculations for the delta-V losses due to gravity and atmospheric drag for a specific altitude, although more sophisticated models can include this to increase accuracy

In [16]:

# ===============================================================
# Simulation and Interpolation Configuration
# ===============================================================
# Example interpolation data for gravitational and atmospheric losses based on LEO altitudes
LEO_ALTITUDES_KM = [200, 300, 400, 500, 600, 700, 800]
DELTA_V_GRAV_KMS = [1520, 1540, 1560, 1580, 1600, 1620, 1640]  # in m/s (example values)
DELTA_V_ATM_KMS = [320, 280, 250, 230, 210, 190, 170]  # in m/s (example values)


# Physical Constants and Ephemeris Setup

Because we will be computing drag, solar radiation, and other types of forces, we define a few constants with the necessary units for future calculations. We also set a starting epoch and initialize two vectors containing the positions of the Sun and the Moon from the starting time until one day later for every half an hour. This is more computationally efficient than calculating the exact position of the celestial bodies for every time step of the simulation.

In [17]:
# ===============================================================
# Global Constants and Ephemeris Setup
# ===============================================================
# Earth-related constants
EARTH_RADIUS_KM = Earth.R.to(u.km).value
EARTH_K = Earth.k.to(u.km**3 / u.s**2).value
DRAG_COEFFICIENT = 2.2  # dimensionless

# Atmospheric model constants (Earth)
SCALE_HEIGHT_KM = H0_earth.to(u.km).value
SEA_LEVEL_DENSITY = rho0_earth.to(u.kg / u.km**3).value
GLOBAL_AREA_TO_MASS = 0  # Placeholder value; updated later in calculation

#Solar radiation constants
rad_coeff = 1.3
Wdivc_s = 3.828e20/299792.458

# Define time epoch and set solar system ephemerides
EPOCH = Time(2451543.597, format="jd", scale="tdb")
solar_system_ephemeris.set("de432s")

# Build ephemeris interpolants for Moon and Sun (half-hour resolution over one day)
start_time = EPOCH
end_time = EPOCH + 1 * u.day
time_step = 0.5 * u.hr

# Calculate the number of points needed to include both start and end times at the given step.
# For 1 day (24h) and 0.5h step: (24/0.5) + 1 = 48 + 1 = 49 points.
num_points = int(round((end_time - start_time).to(u.hr).value / time_step.to(u.hr).value)) + 1
times = start_time + np.arange(num_points) * time_step

moon_ephem = build_ephem_interpolant(
    Moon,
    times, # Changed to positional argument
    attractor=Earth # Explicitly set attractor
)
sun_ephem = build_ephem_interpolant(
    Sun,
    times, # Changed to positional argument
    attractor=Earth # Explicitly set attractor
)


# Initializing the LEO orbit

Here, we initialize the Orbit object that defines our starting LEO orbit. The altitude and the inclination are user-defined, while all other parameters are set to 0.

In [18]:
def create_leo_orbit(leo_altitude_km, inclination_deg):
    """
    Create a circular LEO orbit given altitude and inclination.

    Parameters:
        leo_altitude_km (float): Altitude above the Earth’s surface in km.
        inclination_deg (float): Orbit inclination in degrees.

    Returns:
        Orbit: A poliastro Orbit object created from classical orbital elements.
    """
    semi_major_axis = (leo_altitude_km + 6378) * u.km
    eccentricity = 0.0 * u.one
    inclination = inclination_deg * u.deg
    raan = 0 * u.deg
    arg_periapsis = 0 * u.deg
    true_anomaly = 0 * u.deg

    return Orbit.from_classical(Earth, semi_major_axis, eccentricity,
                                inclination, raan, arg_periapsis, true_anomaly)

# Computing acceleration vector

This function simply computes the accelerations caused by each force on the spacecraft at a specific time. To do this, we use poliastro's built-in functions for each force, the physical constants we defined earlier, and the arrays containing the position of the celestial bodies, as appropiate. It then adds up all the accelerations to get a final acceleration vector and returns this value.

In [19]:
# ===============================================================
# Perturbation and Propagation Functions
# ===============================================================
def perturbing_acceleration(t0, state, gravitational_parameter, J2, J3, radius,
                            drag_coeff, area_to_mass, scale_height, density0, Wdivc_s, rad_coeff, third_body_ephem):
    """
    Compute total perturbing acceleration including J2, J3, atmospheric drag,
    and third-body gravitational effects.
    """
    accel_J2 = J2_perturbation(t0, state, gravitational_parameter, J2, radius)
    accel_drag = atmospheric_drag_exponential(t0, state, gravitational_parameter, radius,
                                              drag_coeff, area_to_mass, scale_height, density0)
    accel_moon = third_body(t0, state, gravitational_parameter,
                            k_third=Moon.k.to(u.km**3 / u.s**2).value,
                            perturbation_body=moon_ephem)
    accel_sun = third_body(t0, state, gravitational_parameter,
                           k_third=Sun.k.to(u.km**3 / u.s**2).value,
                           perturbation_body=third_body_ephem)
    accel_J3 = J3_perturbation(t0, state, gravitational_parameter, J3, radius)
    accel_rad = radiation_pressure(t0, state, gravitational_parameter, radius, rad_coeff, area_to_mass, Wdivc_s, third_body_ephem)

    return accel_J2 + accel_drag + accel_moon + accel_sun + accel_J3

# Computing the derivative of the state vector

This function computes the 2-body Keplerian effect on the spacecraft and adds the total acceleration vector from the other forces to obtain the derivative of the state vector at a specific time. This derivative is a vector containing the velocity and acceleration of the spacecraft in the x, y and z directions. It is used as an indicator of how the position and velocity of the object is chnaging at that point in time.

In [20]:
def total_dynamics(t0, state, gravitational_parameter):
    """
    Compute the total time derivative of the state vector (Keplerian dynamics
    plus perturbations).
    """
    du_kep = func_twobody(t0, state, gravitational_parameter)
    ax, ay, az = perturbing_acceleration(
        t0, state, gravitational_parameter,
        Earth.J2.value, Earth.J3.value,
        EARTH_RADIUS_KM, DRAG_COEFFICIENT, GLOBAL_AREA_TO_MASS,
        SCALE_HEIGHT_KM, SEA_LEVEL_DENSITY, Wdivc_s, rad_coeff, sun_ephem
    )
    perturbation = np.array([0, 0, 0, ax, ay, az])
    return du_kep + perturbation

# Optimizing the rocket's mass

In this simulation, it is assumed that the two-stage rocket provides all the delta-V needed to go from Earth to a GTO orbit, where the payload leaves and circularizes into a GEO orbit. This function looks at the most efficient way to split this delta-V between the two stages of the rocket to minimize its mass.

Based on typical applications and previous literature, we assume the delta-V provided by stage 1 is between 25% to 40% of the total. We test each of the 16 possible scenarios (e.g. going 1% up each time from 25% to 40%) to see which configuration is the most efficient. To do this, we calculate the mass of each stage and add them all up to get the total initial mass. The mass of each stage is calculated using the following equation:
The modified Tsiolkovsky rocket equation accounting for structural mass fraction ($\epsilon$) is:

$$M_{stage} = \frac{M_{stages} \left( e^{\frac{\Delta V}{V_e}} - 1 \right)}{1 - \epsilon e^{\frac{\Delta V}{V_e}}}$$

This is a modified version of Tsiolkovsky rocket equation that accounts for structural mass fraction $\epsilon$. $V_e$ is the exhaust velocity of such stage.  Δ $V$ is the delta-V for that stage. ${M_{stages}}$ represents sum of the masses of each stage that comes after the current stage.

The derivation of this equation can be found on this website: https://space.geometrian.com/calcs/opt-multi-stage.php

In [21]:
def optimize_rocket_mass(exhaust_vel1, exhaust_vel2, satellite_mass, mag_deltaV, deltaV_takeoff):
    """
    Optimize the distribution of delta-V between stage 1 and stage 2 to minimize
    the total rocket mass. The function computes stage masses over a range of splits.

    Parameters:
        exhaust_vel1 (float): Exhaust velocity for the first stage.
        exhaust_vel2 (float): Exhaust velocity for the second stage.
        satellite_mass (float): Dry mass of the satellite.
        mag_deltaV (float): Magnitude of the computed delta-V from orbit propagation (km/s).
        deltaV_takeoff (float): Delta-V required for take-off (km/s).

    Returns:
        tuple: Contains stage1_mass_loss, stage2_mass_loss, wet_stage2_mass,
               deltaV_stage1, deltaV_stage2 corresponding to the optimal split.
    """
    total_required_dv = (mag_deltaV * 1000.0) + deltaV_takeoff
    start_fraction = 0.25

    # Generate candidate splits for stage 1 delta-V allocation (16 candidates)
    deltaV_stage1_candidates = [start_fraction * total_required_dv + i * 0.01 * total_required_dv for i in range(16)]
    deltaV_stage2_candidates = [total_required_dv - dv1 for dv1 in deltaV_stage1_candidates]

    stage1_mass_losses = [None] * 16
    stage2_mass_losses = [None] * 16
    total_rocket_masses = [None] * 16

    # Compute mass requirements for each candidate split using the rocket equation with
    # additional mass fractions (0.06 for stage 1, 0.04 for stage 2)
    for i in range(16):
        stage2_mass_losses[i] = (satellite_mass *
                                 (np.e**(deltaV_stage2_candidates[i] / exhaust_vel2) - 1.0)
                                 / (1.0 - (np.e**(deltaV_stage2_candidates[i] / exhaust_vel2) * 0.04))
        )
        stage1_mass_losses[i] = ((satellite_mass + stage2_mass_losses[i]) *
                                 (np.e**(deltaV_stage1_candidates[i] / exhaust_vel1) - 1.0)
                                 / (1.0 - (np.e**(deltaV_stage1_candidates[i] / exhaust_vel1) * 0.06))
        )
        total_rocket_masses[i] = stage1_mass_losses[i] + stage2_mass_losses[i] + satellite_mass

    # Find candidate index with the minimum total rocket mass
    optimal_index = np.argmin(total_rocket_masses)

    return (
        stage1_mass_losses[optimal_index] * 0.06,
        stage2_mass_losses[optimal_index] * 0.04,
        stage2_mass_losses[optimal_index],
        deltaV_stage1_candidates[optimal_index],
        deltaV_stage2_candidates[optimal_index]
    )


# Propagating the orbit anc calculating delta-V and fuel mass

This function performs all the necessary maneuvers to insert the satellite into GEO at the desired inclination. It begins by computing the velocity of the spacecraft at LEO based on its altitude using the Vis-Viva equation:

$$v = \sqrt{\mu \left( \frac{2}{r} - \frac{1}{a} \right)}$$

where $\mu$ is Newton's gravitational constant times mass of the Earth, $r$ is the distance from the earth, and $a$ is the semi-major axis of the orbit (1 for circular orbits).

This same equation is used to compute the expected velocity of the spacecraft in a GTO orbit at the same distance $r$ from the Earth. The delta-V for insertion into GTO us simply the difference between these two velocities. Trigonometry is used to apply the burn in a direction where the inclination is unchanged. The orbit is then propagated by half of the period of the current GTO orbit

Next, the code performs a combined plane change maneuver where both the inclination and the altitude of the orbit are changed to insert the payload into GEO with a desired inclination. We use the law of cosines to find the magnitude of this delta-V and the law of sines to find how this delta-V should be split between the x, y and z components. The maneuver is performed and the fuel mass for this insertion is calculated using Tsiolkovsky rocket equation.



In [22]:
def propagate_and_compute_transfer_dv(leo_orbit, exhaust_vel1, exhaust_vel2, exhaust_vel_prop,
                                        satellite_mass, deltaV_takeoff, inc_change_percent):
    """
    Propagate the orbit from LEO to a transfer orbit and compute the transfer delta-V
    components along with required propellant loss.

    Parameters:
        leo_orbit (Orbit): Initial LEO orbit.
        exhaust_vel1 (float): Exhaust velocity for the first stage.
        exhaust_vel2 (float): Exhaust velocity for the second stage.
        exhaust_vel_prop (float): Propulsive exhaust velocity.
        satellite_mass (float): Satellite mass.
        deltaV_takeoff (float): Delta-V required for take-off.
        inc_change_percent (float): Percentage factor for inclination change.

    Returns:
        tuple: (magnitude_deltaV, magnitude_plane_change_deltaV, total_propellant_loss)
    """
    # Compute LEO orbital parameters
    leo_semi_major_km = leo_orbit.a.to(u.km).value
    leo_velocity_kms = np.sqrt((Earth.k.value / 1000.0) / leo_semi_major_km) / 1000.0

    # Define GTO parameters from LEO altitude
    gto_semi_major_km = (42164.0 + leo_semi_major_km) / 2.0
    gto_perigee_velocity = np.sqrt((Earth.k.value/1000.0) * ((2.0/leo_semi_major_km) - (1.0/gto_semi_major_km))) / 1000.0
    transfer_deltaV = gto_perigee_velocity - leo_velocity_kms

    # Delta-V components (projected along orbital plane components)
    dv_components = [
        0,
        transfer_deltaV * np.cos(np.deg2rad(leo_orbit.inc.value)),
        transfer_deltaV * np.sin(np.deg2rad(leo_orbit.inc.value))
    ]
    dv_vector = np.array(dv_components) * u.km / u.s

    # Apply first maneuver to raise orbit from LEO to GTO
    impulse_maneuver = Maneuver.impulse(dv_vector)
    gto_orbit = leo_orbit.apply_maneuver(impulse_maneuver)

    # Compute magnitude of applied delta-V (in km/s)
    mag_deltaV = np.linalg.norm(dv_vector.to(u.km/u.s).value)

    # Optimize rocket mass based on stages using an external function below
    stage_results = optimize_rocket_mass(exhaust_vel1, exhaust_vel2, satellite_mass,
                                         mag_deltaV, deltaV_takeoff)
    stage1_mass_loss, stage2_mass_loss, wet_stage2_mass, deltaV_stage1, deltaV_stage2 = stage_results

    # Compute updated area-to-mass ratio for drag calculations
    local_area_to_mass = ((((np.pi) * (1.85**2)) * (u.m**2)) /
                          ((satellite_mass + stage2_mass_loss) * u.kg)
                          ).to(u.km**2 / u.kg).value

    # Propagate the orbit until apogee (half period propagation)
    time_to_apogee = (gto_orbit.period.value / 2.0) * u.s
    propagated_orbit = gto_orbit.propagate(time_to_apogee, method=CowellPropagator(f=total_dynamics))

    # Calculate velocity for GEO based on norm of propagated position vector
    norm_radius = np.linalg.norm(propagated_orbit.r.value)
    geo_velocity = np.sqrt(Earth.k.value / 1000.0 * ((2.0/norm_radius) - (1.0/42164.0))) / 1000.0

    # Cancel x-component of velocity due to perturbations with a second burn
    dv_cancel = np.array([-propagated_orbit.v.value[0], 0, 0]) * (u.m / u.s)
    impulse_cancel = Maneuver.impulse(dv_cancel)
    propagated_orbit_corrected = propagated_orbit.apply_maneuver(impulse_cancel)

    # Compute delta-V for second burn (aiming to circularize GEO orbit)
    mag_deltaV_plane = np.sqrt(np.linalg.norm(propagated_orbit_corrected.v.value)**2 +
                               geo_velocity**2 -
                               2 * geo_velocity *
                               np.linalg.norm(propagated_orbit_corrected.v.value) *
                               np.cos(propagated_orbit_corrected.inc.value *
                                      (inc_change_percent/100)))

    # Determine flight path angle based on circularization maneuver requirements
    flight_path_angle_deg = (180 - np.degrees(np.arcsin((geo_velocity *
                                 np.sin(propagated_orbit_corrected.inc.value * (inc_change_percent/100)) /
                                 mag_deltaV_plane))))

    # Calculate the components of the delta-V vector using different approaches based on inclination
    if leo_orbit.inc.value >= 59 and inc_change_percent == 100:
        vector1_magnitude = np.abs(mag_deltaV_plane * np.cos(np.radians(flight_path_angle_deg - 90)))
    else:
        vector1_magnitude = np.abs(mag_deltaV_plane * np.cos(np.radians(flight_path_angle_deg)))

    angle_vector1 = np.arctan2(propagated_orbit_corrected.v.value[2],
                               propagated_orbit_corrected.v.value[1])
    vector1 = [0, vector1_magnitude * np.cos(angle_vector1),
                  vector1_magnitude * np.sin(angle_vector1)]

    if leo_orbit.inc.value >= 59 and inc_change_percent == 100:
        vector2_magnitude = np.abs(mag_deltaV_plane * np.sin(np.radians(flight_path_angle_deg - 90)))
    else:
        vector2_magnitude = np.abs(mag_deltaV_plane * np.sin(np.radians(flight_path_angle_deg)))

    # Calculate perpendicular vector to vector1 in the orbital plane
    vector1_perp = [0, propagated_orbit_corrected.v.value[2], -propagated_orbit_corrected.v.value[1]]
    angle_vector2 = np.arctan2(vector1_perp[2], vector1_perp[1])
    vector2 = [0, vector2_magnitude * np.cos(angle_vector2),
                  vector2_magnitude * np.sin(angle_vector2)]

    # Combine vectors to produce the required GEO plane-change maneuver
    if leo_orbit.inc.value >= 59 and inc_change_percent == 100:
        combined_dv_vector = [0, -(vector1[2] + vector2[2]),
                                vector1[1] + vector2[1]]
    else:
        combined_dv_vector = [0, -(vector1[1] + vector2[1]),
                                -(vector1[2] + vector2[2])]

    impulse_plane_change = Maneuver.impulse(np.array(combined_dv_vector) * (u.km/u.s))
    geo_orbit = propagated_orbit_corrected.apply_maneuver(impulse_plane_change)

    # Total combined delta-V for plane change maneuver (including cancellation burn)
    combined_plane_dv = mag_deltaV_plane + np.abs(propagated_orbit.v.value[0])

    # Propellant loss calculations using the rocket equation (staged)
    loss_stage3 = satellite_mass - (satellite_mass / np.e**((mag_deltaV_plane * 1000.0) / exhaust_vel_prop))
    loss_stage2 = (satellite_mass + stage2_mass_loss) * (np.e**((mag_deltaV * 1000.0) / (348.0 * 9.8)) - 1.0)
    loss_stage1 = (satellite_mass + stage2_mass_loss + loss_stage2) * (np.e**((deltaV_stage2 - (mag_deltaV * 1000.0)) / (348.0 * 9.8)) - 1.0)
    loss_initial = (satellite_mass + wet_stage2_mass + stage1_mass_loss) * (np.e**(deltaV_stage1 / (300.0 * 9.8)) - 1.0)
    total_propellant_loss = loss_initial + loss_stage1 + loss_stage2 + loss_stage3

    return mag_deltaV, combined_plane_dv, total_propellant_loss

# Simulating the orbit

This function utilizes all the functions described above to obtain the total fuel mass and delta-V for all maneuevers. It also computes an approximate of the delta-V required to insert the rocket into LEO orbit, accounting for drag, gravity, earth's boost, etc.

In [23]:
# ===============================================================
# Simulation Functions
# ===============================================================
def simulate_orbit(sim_params):
    """
    Perform a single orbit simulation.

    Parameters:
        sim_params (tuple): Contains the following parameters:
            - inclination_deg: LEO orbital inclination in degrees.
            - leo_altitude_km: LEO altitude above Earth surface (km).
            - exhaust_vel_stage1: Exhaust velocity for stage 1.
            - exhaust_vel_stage2: Exhaust velocity for stage 2.
            - exhaust_vel_propulsive: Propulsive exhaust velocity.
            - satellite_mass: Dry mass of satellite.
            - inclination_change_percent: Percent value for inclination change.

    Returns:
        tuple: (inclination_deg, inclination_change_percent, leo_altitude_km,
                exhaust_vel_stage1, exhaust_vel_stage2, exhaust_vel_propulsive,
                satellite_mass, f_total, total_deltaV)
    """
    # Unpack simulation parameters
    (inclination_deg, leo_altitude_km,
     exhaust_vel_stage1, exhaust_vel_stage2, exhaust_vel_propulsive,
     satellite_mass, inc_change_percent) = sim_params

    # Create an initial LEO orbit object using given altitude and inclination
    leo_orbit = create_leo_orbit(leo_altitude_km, inclination_deg)

    # Compute delta-V components from orbital mechanics and perturbations
    transfer_dv = np.sqrt(Earth.k.value / (6378000 + (leo_altitude_km * 1000)))
    potential_dv = (np.sqrt(Earth.k.value / 6378000) *
                    np.sqrt(2 - (6378000 / (6378000 + (leo_altitude_km * 1000)))) -
                    np.sqrt(Earth.k.value / (6378000 + (leo_altitude_km * 1000))))
    grav_loss_dv = np.interp(leo_altitude_km, LEO_ALTITUDES_KM, DELTA_V_GRAV_KMS)
    atm_loss_dv = np.interp(leo_altitude_km, LEO_ALTITUDES_KM, DELTA_V_ATM_KMS)
    earth_boost_dv = (7.2921159e-5) * Earth.R.to(u.m).value * np.cos(leo_orbit.inc.to(u.rad).value)

    # Total take-off delta-V is the sum of various components minus the boost provided by Earth's rotation
    deltaV_takeoff = transfer_dv + grav_loss_dv + potential_dv + atm_loss_dv - earth_boost_dv

    # Compute additional delta-V contributions by propagating the orbit and other calculations.
    mag_deltaV, mag_deltaV_plane, total_propulsive_loss = propagate_and_compute_transfer_dv(
        leo_orbit, exhaust_vel_stage1, exhaust_vel_stage2, exhaust_vel_propulsive,
        satellite_mass, deltaV_takeoff, inc_change_percent
    )

    total_deltaV = deltaV_takeoff + mag_deltaV * 1000 + mag_deltaV_plane * 1000

    # Return complete simulation results
    return (inclination_deg, inc_change_percent, leo_altitude_km,
            exhaust_vel_stage1, exhaust_vel_stage2, exhaust_vel_propulsive,
            satellite_mass, total_propulsive_loss, total_deltaV)